# Visualise the neighbour-exchange channel

Self-contained. Set the base potential and one gate geometry below, then run all.
The gate parameter reported by `search.py` is `n_delta = 1 - n_target`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from vbc_landscape import swap_landscape, find_swap_endpoints, minimax_barrier

# base potential
R_REP, R_ATT, E_R, E_A = 1.0, 1.5, 20.0, -2.0

# one gate geometry to look at  (r_low, r_high, n_target);  n_delta = 1 - n_target
R_LOW, R_HIGH, N_TARGET = 1.29, 1.50, 0.70

VALENCIES = [2, 4, 6]
r_grid = np.linspace(0.9, 2.2, 401)

In [ ]:
def draw_channel(ax, valency):
    U = swap_landscape(valency, r_grid, R_LOW, R_HIGH, N_TARGET,
                       r_rep=R_REP, r_att=R_ATT, E_r=E_R, E_a=E_A)
    ext = [r_grid[0], r_grid[-1], r_grid[0], r_grid[-1]]
    im = ax.imshow((U / abs(E_A)).T, origin="lower", extent=ext, aspect="equal",
                   vmin=-valency, vmax=0.0, cmap="viridis")
    ax.set_xlabel(r"$r_A$")
    ax.set_ylabel(r"$r_B$")
    title = f"f = {valency}"
    ep = find_swap_endpoints(U, r_grid)
    if ep is not None:
        B = minimax_barrier(U, ep[0], ep[1])
        B_rel = B - max(U[ep[0]], U[ep[1]])
        ax.contour(r_grid, r_grid, U.T, levels=[B], colors="r", linewidths=1.0)
        for (i, j) in ep:
            ax.plot(r_grid[i], r_grid[j], "r*", ms=13)
        title += f"    B/|E_a| = {B_rel / abs(E_A):.3f}"
    ax.set_title(title)
    return im

fig, axes = plt.subplots(1, len(VALENCIES), figsize=(4.2 * len(VALENCIES), 4.0))
axes = np.atleast_1d(axes)
im = None
for ax, v in zip(axes, VALENCIES):
    im = draw_channel(ax, v)
fig.colorbar(im, ax=list(axes), label=r"$U / |E_a|$", shrink=0.85)
plt.show()

## Read back a search result

Point at a `top*_barrier.csv` written by `search.py`.

In [ ]:
import csv, os

path = "./outputs/top100_barrier.csv"
if os.path.exists(path):
    with open(path) as f:
        best = next(csv.DictReader(f))
    print("best gate   r_low=%s  r_high=%s  n_target=%s  (n_delta=%s)"
          % (best["r_low"], best["r_high"], best["n_target"], best["n_delta"]))
    print("worst-case  B/|E_a| = %s    spur/|E_a| = %s"
          % (best["bEa_max"], best["spur_max"]))
    fs = sorted(int(k[5:]) for k in best if k.startswith("bEa_v"))
    bvals = [float(best[f"bEa_v{f}"]) for f in fs]
    fig, ax = plt.subplots(figsize=(4.0, 3.0))
    ax.plot(fs, bvals, "o-")
    ax.set_xlabel("valency f")
    ax.set_ylabel(r"$B / |E_a|$")
    ax.set_title("exchange barrier of the best gate")
    plt.show()
else:
    print("Run search.py first, e.g.  python search.py --out_dir ./outputs")

## Spurious (off-channel) minima

The profile is the total energy of a particle approaching an already-saturated
neighbour, including the back-reaction on the neighbour's existing bonds. It is drawn
relative to the N-far baseline, so a dip below zero is a transient trap and its depth
is the value returned by `sporadic_min_depth`.

In [ ]:
from vbc_landscape import density_phi, omega_gate, V_rep, V_att, sporadic_min_depth

def spurious_profile(valency, r_low, r_high, n_target, *, r_rep, r_att, E_r, E_a, n_r=501):
    r = np.linspace(r_rep, r_att, n_r)
    ncut = valency + (n_target - 1.0)
    r_star = 0.5 * (r_rep + r_att)
    rho_rs = float(density_phi(np.array([r_star]), r_low, r_high)[0])
    vatt_rs = float(V_att(np.array([r_star]), r_rep, r_att, E_a)[0])
    W_S = float(omega_gate(np.array([rho_rs]), ncut)[0])
    rho_r = density_phi(r, r_low, r_high)
    W_C = omega_gate(valency * rho_rs + rho_r, ncut)
    W_N = omega_gate(rho_r, ncut)
    U = (V_rep(r, r_rep, E_r) + W_C * W_N * V_att(r, r_rep, r_att, E_a)
         + valency * W_C * W_S * vatt_rs)
    baseline = valency * float(omega_gate(np.array([valency * rho_rs]), ncut)[0]) * W_S * vatt_rs
    return r, U, baseline

def draw_spurious(ax, name, gate, valencies):
    r_low, r_high, n_target = gate
    for f in valencies:
        r, U, base = spurious_profile(f, r_low, r_high, n_target,
                                      r_rep=R_REP, r_att=R_ATT, E_r=E_R, E_a=E_A)
        d = sporadic_min_depth(f, r_low, r_high, n_target,
                               r_rep=R_REP, r_att=R_ATT, E_r=E_R, E_a=E_A)
        ax.plot(r, (U - base) / abs(E_A), label=f"f={f}   depth={d:.3f}")
    ax.axhline(0.0, color="k", lw=0.8, ls="--")
    ax.set_xlabel(r"$r_{CN}$")
    ax.set_ylabel(r"$(U - U_\infty) / |E_a|$")
    ax.set_title(name)
    ax.legend(fontsize=7)

In [ ]:
# the three gates from the RCS paper, on the shared base potential above
# VBC(r_low, r_high, n_delta);  n_target = 1 - n_delta
SETUPS = [
    ("standard  VBC(1.5, 2.0, 0)",         (1.5,   2.0,   1.00)),
    ("looser  VBC(1.224, 1.561, -0.03)",   (1.224, 1.561, 1.03)),
    ("optimised  VBC(1.29, 1.50, 0.30)",   (1.29,  1.50,  0.70)),
]
VALS = [2, 3, 4, 5, 6]

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0), sharey=True)
for ax, (name, gate) in zip(axes, SETUPS):
    draw_spurious(ax, name, gate, VALS)
plt.tight_layout()
plt.show()

## Random passive placement

With `key="random"` the passive neighbours are scattered uniformly in the binding
zone rather than sitting at the well centre, so each contributes its own density
weight to the count. The depth is averaged over `n_sample` placements. Note that a
random placement need not keep the centre saturated, so the mean blends genuine
binding with true spurious traps.

In [ ]:
from vbc_landscape import spurious_depth

N_SAMPLE = 200
fig, ax = plt.subplots(figsize=(5.5, 3.8))
for name, gate in SETUPS:
    rl, rh, nt = gate
    avg = [spurious_depth(f, rl, rh, nt, r_rep=R_REP, r_att=R_ATT, E_r=E_R, E_a=E_A,
                          key="random", n_sample=N_SAMPLE, seed=0) for f in VALS]
    ax.plot(VALS, avg, "o-", label=name)
    print(name, " ", "  ".join(f"f{f}={a:.3f}" for f, a in zip(VALS, avg)))
ax.set_xlabel("valency f")
ax.set_ylabel(r"mean spurious depth $/|E_a|$")
ax.set_title(f"random passive placement, mean over n_sample={N_SAMPLE}")
ax.legend(fontsize=7)
plt.show()

## Stability analysis

The off-channel analysis in increasing fidelity, all on the same base potential.

- 1 Naive. Bare C-N pair, no back-reaction, baseline zero.
- 2 Deterministic. Total cluster energy, all passive neighbours at the well centre.
- 3 Random. Total cluster energy, neighbours scattered in the binding zone, averaged.
- 4 Local Hessian stability. Curvature of the total energy at sampled arrangements (needs JAX).



In [ ]:
from vbc_landscape import naive_spurious_depth, sporadic_min_depth, spurious_depth

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0), sharey=True)
for ax, (name, gate) in zip(axes, SETUPS):
    rl, rh, nt = gate
    kw = dict(r_rep=R_REP, r_att=R_ATT, E_r=E_R, E_a=E_A)
    naive = [naive_spurious_depth(f, rl, rh, nt, **kw) for f in VALS]
    det   = [sporadic_min_depth(f, rl, rh, nt, **kw) for f in VALS]
    rand  = [spurious_depth(f, rl, rh, nt, key="random", n_sample=200, seed=0, **kw) for f in VALS]
    ax.plot(VALS, naive, "o-", label="L1 naive")
    ax.plot(VALS, det,   "s-", label="L2 deterministic")
    ax.plot(VALS, rand,  "^-", label="L3 random")
    ax.set_xlabel("valency f")
    ax.set_title(name)
    ax.legend(fontsize=7)
axes[0].set_ylabel(r"spurious depth $/|E_a|$")
plt.tight_layout()
plt.show()

## Local Hessian stability

For each neighbour arrangement the Hessian of the total energy over the neighbour radii
is diagonalised. `p_stable` is the fraction of arrangements whose Hessian is
positive-definite (locally convex), and `lambda_min/kappa0` is the softest curvature,
with `kappa0 = 16|E_a|/(r_att-r_rep)^2`. A sampled arrangement is generally not a
stationary point (grad != 0), so this is local convexity, not a relaxed basin minimum.
Install JAX to run these cells (see README). 

In [ ]:
try:
    import jax  # noqa: F401
    from stability_jax import stability_random, curvature_scale
    HAVE_JAX = True
except ModuleNotFoundError:
    HAVE_JAX = False
    print("JAX not installed - skipping Layer 4.  pip install 'jax[cpu]'  (see README).")

if HAVE_JAX:
    k0 = curvature_scale(E_A, R_REP, R_ATT)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 4.0))
    for name, gate in SETUPS:
        rl, rh, nt = gate
        ps = [stability_random(f, rl, rh, nt, r_rep=R_REP, r_att=R_ATT, E_r=E_R, E_a=E_A,
                               n_sample=20000, seed=0).p_stable for f in VALS]
        ax1.plot(VALS, ps, "o-", label=name)
    ax1.set_xlabel("valency f")
    ax1.set_ylabel("p_stable (Hessian positive-definite)")
    ax1.set_title("fraction of arrangements locally convex")
    ax1.legend(fontsize=7)
    for name, gate in SETUPS:
        rl, rh, nt = gate
        r = stability_random(6, rl, rh, nt, r_rep=R_REP, r_att=R_ATT, E_r=E_R, E_a=E_A,
                             n_sample=20000, seed=0)
        ax2.hist(r.lambda_min / k0, bins=60, histtype="step", density=True, label=name)
    ax2.axvline(0.0, color="k", lw=0.8, ls="--")
    ax2.set_xlabel(r"$\lambda_{\min}/\kappa_0$   (f=6)")
    ax2.set_ylabel("density")
    ax2.set_title("softest-mode curvature")
    ax2.legend(fontsize=7)
    plt.tight_layout()
    plt.show()

In [ ]:
if HAVE_JAX:
    import time
    from stability_jax import stability_grid
    print("deduped radial grid (n_grid=10), p_stable over distinct arrangements:")
    for name, gate in SETUPS:
        rl, rh, nt = gate
        cols = []
        for f in [4, 6]:
            r = stability_grid(f, rl, rh, nt, r_rep=R_REP, r_att=R_ATT, E_r=E_R, E_a=E_A,
                               n_grid=10, dedup=True)
            cols.append(f"f{f}: n={r.n_config} p_stable={r.p_stable:.3f}")
        print(f"  {name}:  " + "   ".join(cols))
    # throughput benchmark on the full un-deduped 1e6 grid (valency 5 -> m=6)
    t = time.perf_counter()
    r = stability_grid(5, 1.29, 1.50, 0.70, r_rep=R_REP, r_att=R_ATT, E_r=E_R, E_a=E_A,
                       n_grid=10, dedup=False, chunk=100000)
    dt = time.perf_counter() - t
    print(f"benchmark: full {r.n_config:,} config grid (m={r.m}) in {dt:.1f}s"
          f"  ->  {r.n_config/dt:,.0f} Hessian diagonalisations/s")

## Layer 4 (rigorous): over-coordination traps

The stability tests use `m = valency + 1` neighbours, the saturating shell plus one extra
approaching particle. A spurious trap is a strictly convex interior minimum of the total
energy (grad U = 0, Hessian positive-definite) in which all `valency + 1` neighbours stay
bonded. `find_minima` relaxes seeds to grad U = 0 and keeps only such minima (the flat
U = 0 plateau of a gate that switches every bond off is excluded, it is not a trap).

In [ ]:
if HAVE_JAX:
    import jax.numpy as jnp
    from vbc_landscape_jax import cluster_energy
    f = 6; m = f + 1
    rgrid = np.linspace(R_REP, R_ATT, 400)
    fig, ax = plt.subplots(figsize=(6.2, 4.0))
    for name, gate in SETUPS:
        rl, rh, nt = gate; ncut = f + (nt - 1.0)
        U = [float(cluster_energy(jnp.full(m, r), r_low=rl, r_high=rh, ncut=ncut,
                                  r_rep=R_REP, r_att=R_ATT, E_r=E_R, E_a=E_A)) for r in rgrid]
        ax.plot(rgrid, np.array(U) / abs(E_A), label=name)
    ax.axhline(0.0, color="k", lw=0.8, ls="--")
    ax.set_xlabel(r"$r$   (all $m=f{+}1$ neighbours at radius $r$)")
    ax.set_ylabel(r"$U/|E_a|$")
    ax.set_title(f"over-coordinated symmetric energy (valency f={f}, m={m})")
    ax.legend(fontsize=7)
    plt.show()

In [ ]:
if HAVE_JAX:
    from stability_jax import find_minima, curvature_scale
    k0 = curvature_scale(E_A, R_REP, R_ATT)
    print("strictly convex interior minima of the over-coordinated cluster (m = valency+1):")
    for name, gate in SETUPS:
        rl, rh, nt = gate
        for f in VALS:
            mins = find_minima(f, rl, rh, nt, r_rep=R_REP, r_att=R_ATT, E_r=E_R, E_a=E_A,
                               n_seed=100)
            if mins:
                x, u, lmin = mins[0]
                print(f"  {name} f={f}: TRAP  r~{x[0]:.3f}  U/|E_a|={u/abs(E_A):+.3f}"
                      f"  lam_min/k0={lmin/k0:+.3f}  ({len(mins)} distinct)")
            else:
                print(f"  {name} f={f}: none (extra particle rejected)")

### How deep is the looser-gate trap

The energy of the over-coordinated `(f+1)` trap relative to the ideal `f`-coordinated
state (all `f` bonds satisfied, `U = -f|E_a|`, which is also the standard gate's `f`
optimum). A negative depth means the extra particle is net bound, so over-coordination
lies below the ideal `f` state. a positive depth is a metastable pocket above it.

In [ ]:
if HAVE_JAX:
    from stability_jax import find_minima, optimal_state_energy
    std_gate = SETUPS[0][1]      # standard VBC(1.5, 2.0, 0)
    looser_gate = SETUPS[1][1]   # looser  VBC(1.224, 1.561, -0.03)
    kw = dict(r_rep=R_REP, r_att=R_ATT, E_r=E_R, E_a=E_A)
    depth = []
    print(f"  {'f':>2}  {'U_trap(f+1)/|E_a|':>17}  {'std ideal f/|E_a|':>17}  {'depth/|E_a|':>12}")
    for f in VALS:
        Uref = optimal_state_energy(f, *std_gate, **kw) / abs(E_A)
        mins = find_minima(f, *looser_gate, n_seed=100, **kw)
        if not mins:
            depth.append(np.nan)
            print(f"  {f:>2}  {'(no trap)':>17}  {Uref:>17.3f}")
            continue
        Ut = mins[0][1] / abs(E_A)
        depth.append(Ut - Uref)
        print(f"  {f:>2}  {Ut:>17.3f}  {Uref:>17.3f}  {Ut-Uref:>+12.3f}")
    fig, ax = plt.subplots(figsize=(5.5, 3.6))
    ax.plot(VALS, depth, "o-")
    ax.axhline(0.0, color="k", lw=0.8, ls="--")
    ax.set_xlabel("valency f")
    ax.set_ylabel(r"$(U_{\rm trap}(f{+}1)-U_{\rm ideal}(f))/|E_a|$")
    ax.set_title("looser gate: depth of the over-coordinated trap")
    plt.show()